In [1]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

In [2]:
WINDOWS = sys.platform.startswith("win")

# Required BLAST + DB location
DEFAULT_INSTALL_DIR = r"C:\BLAST"

# Common BLAST install locations (bin folder)
COMMON_BLAST_PATHS = [
    r"C:\BLAST\bin",  # 👈 add this FIRST (your requirement)
    r"C:\Program Files\NCBI\blast\bin",
    r"C:\Program Files\NCBI\blast-2.14.0+\bin",
    r"C:\Program Files\NCBI\blast-2.13.0+\bin",
    r"C:\Program Files\NCBI\blast-2.12.0+\bin",
    r"C:\Program Files (x86)\NCBI\blast\bin",
]

In [3]:
def run_cmd(command, capture_output=True):
    """Run a command and return a completed process."""
    return subprocess.run(
        command,
        shell=isinstance(command, str),
        capture_output=capture_output,
        text=True,
    )

In [4]:
def is_windows():
    if not WINDOWS:
        print("❌ ERROR: This pipeline only supports Windows.")
        print("👉 Please run on a Windows machine.")
        return False
    return True

In [5]:
def find_blast():

    print("\n🔍 Checking BLAST installation...")

    # First: check if already in PATH
    blastp = shutil.which("blastp")

    if blastp:
        print(f"✅ BLAST found in PATH: {blastp}")
        return blastp

    # Second: check common install folders
    for path in COMMON_BLAST_PATHS:
        exe = Path(path) / "blastp.exe"
        if exe.exists():
            print(f"✅ BLAST found at: {exe}")
            return str(exe)

    # If not found
    print("\n❌ ERROR: BLAST not found.")
    print("👉 Please install NCBI BLAST+")
    print("👉 Expected location: C:\\BLAST\\bin")
    print("👉 Download from: https://blast.ncbi.nlm.nih.gov/Blast.cgi?PAGE_TYPE=BlastDocs&DOC_TYPE=Download")

    return None

In [6]:
def check_blast_folder():

    print("\n📁 Checking C:\\BLAST folder...")

    folder = Path(DEFAULT_INSTALL_DIR)

    if not folder.exists():
        print("❌ ERROR: C:\\BLAST folder not found")
        print("👉 Please create C:\\BLAST and place your database there")
        return False

    print("✅ C:\\BLAST folder exists")
    return True

In [7]:
def find_databases():

    print("\n🔍 Checking BLAST databases...")

    folder = Path(DEFAULT_INSTALL_DIR)

    db_files = [".pin", ".psq", ".phr"]
    databases = set()

    for file in folder.iterdir():
        if file.suffix.lower() in db_files:
            databases.add(file.stem)

    if not databases:
        print("❌ ERROR: No BLAST protein database found")
        print("👉 Place database files (.pin/.psq/.phr) in C:\\BLAST")
        return None

    print("✅ Found databases:")
    for db in databases:
        print(f"   - {db}")

    return list(databases)

In [8]:
if not is_windows():
    sys.exit()

blast = find_blast()
if not blast:
    sys.exit()

if not check_blast_folder():
    sys.exit()

databases = find_databases()
if not databases:
    sys.exit()

print("\n🎉 All checks passed! Ready to run BLAST.")


🔍 Checking BLAST installation...
✅ BLAST found in PATH: C:\Users\nurly\OneDrive\Documents\UKM\BioHackathon\blast-2.17.0+\bin\blastp.EXE

📁 Checking C:\BLAST folder...
✅ C:\BLAST folder exists

🔍 Checking BLAST databases...
✅ Found databases:
   - swissprot

🎉 All checks passed! Ready to run BLAST.


In [9]:
def install_blast_via_winget():

    print("\n🔍 Checking winget...")

    winget = shutil.which("winget")

    if not winget:
        print("❌ winget not found")
        print("👉 Install Microsoft App Installer first")
        return False


    packages = [
        "NCBI.NCBIblast",
        "NCBI.BLAST",
        "NCBI.NCBIBLAST",
        "NCBI.Blast"
    ]


    for package in packages:

        print(f"\nTrying package: {package}")

        result = run_cmd(
            f'winget install --id "{package}" '
            '--accept-source-agreements '
            '--accept-package-agreements '
            '--silent'
        )


        if result.returncode == 0:

            print("✅ BLAST installed successfully")
            return True


        else:

            print("❌ Package failed")

            if result.stderr:
                print(result.stderr)


    print("\n❌ Automatic BLAST installation failed")
    return False

In [10]:
def install_blast_via_choco():

    print("\n🔍 Checking Chocolatey...")


    choco = shutil.which("choco")


    if not choco:
        print("❌ Chocolatey not found")
        return False


    result = run_cmd(
        "choco install ncbi-blast -y"
    )


    if result.returncode == 0:

        print("✅ BLAST installed using Chocolatey")
        return True


    print("❌ Chocolatey installation failed")

    return False

In [11]:
def install_blast():

    print("\n==========================")
    print("Checking BLAST installation")
    print("==========================")


    blast = find_blast()


    if blast:

        print("✅ BLAST already installed")
        print(blast)
        return True



    print("\n❌ BLAST not found")
    print("Expected location:")
    print(r"C:\BLAST\bin")


    choice = input(
        "\nInstall BLAST now? (Y/N): "
    ).lower()



    if choice not in ["y","yes"]:

        print("Installation cancelled")
        return False



    if install_blast_via_winget():

        return True



    if install_blast_via_choco():

        return True



    print("\n❌ Could not install BLAST")
    print("Please install manually from NCBI")

    return False